# GroupDNA — WhatsApp Group Chat Analyzer
Name:Romi kuamar singh
Batch: july
Date: 25-07-2026

## Feature 1: The Chat Parser

In [1]:
import numpy as np
from datetime import datetime, timedelta

FILENAME = "DADS Minor PROJECT dataset.txt"   # adjust path if needed

def is_date_start(line):
    # crude check: do the first 8 characters look like DD/MM/YY ?
    if len(line) < 8:
        return False
    d = line[:8]
    return (d[0:2].isdigit() and d[2] == '/' and d[3:5].isdigit()
            and d[5] == '/' and d[6:8].isdigit())

def parse_chat(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        raw_lines = f.read().split('\n')

    messages = []          # list of dicts: timestamp, sender, text, type
    system_count = 0
    media_count = 0
    deleted_count = 0
    current = None          # holds the message currently being built (for multi-line continuations)

    for line in raw_lines:
        if line.strip() == '':
            continue   # skip empty lines silently

        if not is_date_start(line):
            # this line doesn't start with a date -> it's a continuation of the previous message
            if current is not None:
                current['text'] += ' ' + line.strip()
            continue

        # a new dated line begins -> flush whatever message we were building
        if current is not None:
            messages.append(current)
            current = None

        # split "timestamp - rest"
        parts = line.split(' - ', 1)
        if len(parts) != 2:
            system_count += 1
            continue
        timestamp_str, rest = parts

        # split "sender: message"
        rest_parts = rest.split(': ', 1)
        if len(rest_parts) != 2:
            # no colon after a name -> WhatsApp system message (group created, added, etc.)
            system_count += 1
            continue

        sender, text = rest_parts

        if text.strip() == 'This message was deleted':
            deleted_count += 1
            current = {'timestamp': timestamp_str, 'sender': sender, 'text': text, 'type': 'deleted'}
            continue

        if text.strip() == '<Media omitted>':
            media_count += 1
            current = {'timestamp': timestamp_str, 'sender': sender, 'text': text, 'type': 'media'}
            continue

        current = {'timestamp': timestamp_str, 'sender': sender, 'text': text, 'type': 'real'}

    if current is not None:
        messages.append(current)

    return messages, system_count, media_count, deleted_count


def parse_dt(ts):
    return datetime.strptime(ts, '%d/%m/%y, %H:%M')


messages, system_count, media_count, deleted_count = parse_chat(FILENAME)
for m in messages:
    m['dt'] = parse_dt(m['timestamp'])

real_messages = [m for m in messages if m['type'] == 'real']
participants = sorted(set(m['sender'] for m in messages))

print(f"Successfully parsed {len(messages)} messages from {len(participants)} participants, "
      f"skipped {system_count} system messages, {media_count} media-omitted, {deleted_count} deleted messages.")
print(f"Real (analyzable) messages: {len(real_messages)}")
print("\nFirst 5 messages:")
for m in messages[:5]:
    print(" ", m['timestamp'], '-', m['sender'], ':', m['text'])
print("\nLast 5 messages:")
for m in messages[-5:]:
    print(" ", m['timestamp'], '-', m['sender'], ':', m['text'])

Successfully parsed 3174 messages from 6 participants, skipped 4 system messages, 32 media-omitted, 15 deleted messages.
Real (analyzable) messages: 3127

First 5 messages:
  01/04/24, 01:17 - Rahul : scene fix
  01/04/24, 01:17 - Rahul : haan
  01/04/24, 01:18 - Rahul : kya scene
  01/04/24, 02:13 - Rahul : abhi free hai?
  01/04/24, 02:13 - Rahul : abey

Last 5 messages:
  30/05/24, 19:14 - Priya : Take care everyone
  30/05/24, 19:28 - Priya : Karan that sounds tough, take care
  30/05/24, 21:17 - Aman : the existential dread is back
  30/05/24, 21:30 - Karan : Long day guys, woke up at six for that placement workshop which started at eight by the way classic, then ran around campus collecting signatures for the project document, ate lunch at four PM standing in the corridor, came back to hostel and realized I forgot my charger in the lab.
  30/05/24, 23:31 - Aman : anyone awake?


## Feature 2: Group Overview

In [2]:
# Group-level headline stats: date range, total messages, per-person breakdown

start_date = min(m['dt'] for m in messages)
end_date = max(m['dt'] for m in messages)
total_days = (end_date.date() - start_date.date()).days + 1

# count ALL message types per person (media/deleted still count as "sent something")
person_counts = {}
for m in messages:
    person_counts[m['sender']] = person_counts.get(m['sender'], 0) + 1

total_msgs = sum(person_counts.values())
sorted_people = sorted(person_counts.items(), key=lambda x: x[1], reverse=True)

print("=" * 60)
print(" GROUP OVERVIEW".center(60))
print("=" * 60)
print(f" Period       : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')} ({total_days} days)")
print(f" Total messages : {total_msgs}")
print(f" Participants  : {len(participants)}")
print("\n MESSAGES PER PERSON")
for name, count in sorted_people:
    pct = count / total_msgs * 100
    print(f"   {name:<10}: {count:>4} ({pct:5.1f}%)")

                       GROUP OVERVIEW                       
 Period       : 01 April 2024 to 30 May 2024 (60 days)
 Total messages : 3174
 Participants  : 6

 MESSAGES PER PERSON
   Rahul     :  953 ( 30.0%)
   Priya     :  718 ( 22.6%)
   Neha      :  635 ( 20.0%)
   Aman      :  490 ( 15.4%)
   Karan     :  354 ( 11.2%)
   Vikas     :   24 (  0.8%)


## Feature 3: Most Active Day and Hour

In [3]:
# Which single day, and which hour of the day, had the most messages overall

day_counts = {}
hour_counts = {}
for m in messages:
    day_key = m['dt'].date()
    day_counts[day_key] = day_counts.get(day_key, 0) + 1
    hour_counts[m['dt'].hour] = hour_counts.get(m['dt'].hour, 0) + 1

busiest_day = max(day_counts.items(), key=lambda x: x[1])
busiest_hour = max(hour_counts.items(), key=lambda x: x[1])

print(" ACTIVITY PEAKS")
print(f"   Busiest day  : {busiest_day[0].strftime('%d %B %Y')} ({busiest_day[1]} messages)")
print(f"   Busiest hour : {busiest_hour[0]:02d}:00 - {(busiest_hour[0]+1)%24:02d}:00 ({busiest_hour[1]} messages)")

 ACTIVITY PEAKS
   Busiest day  : 04 May 2024 (76 messages)
   Busiest hour : 18:00 - 19:00 (248 messages)


## Feature 4: Activity Heatmap (NumPy)

In [4]:
# Build a 6x24 NumPy matrix: rows = participants, columns = hour of day (0-23)
# Each cell = how many messages that person sent during that hour, across all 60 days

person_index = {name: i for i, name in enumerate(participants)}
heatmap = np.zeros((len(participants), 24), dtype=int)

for m in messages:
    row = person_index[m['sender']]
    col = m['dt'].hour
    heatmap[row, col] += 1

print("Heatmap matrix shape:", heatmap.shape)
print(heatmap)

print("\n ACTIVITY HEATMAP (messages by hour)")
print("          " + " ".join(f"{h:02d}" for h in range(0, 24, 3)))

for name, idx in person_index.items():
    row = heatmap[idx]
    row_max = row.max() if row.max() > 0 else 1
    symbols = []
    for h in range(0, 24, 3):
        ratio = row[h] / row_max
        if ratio <= 0.25:
            symbols.append('.  ')
        elif ratio <= 0.5:
            symbols.append('░  ')
        elif ratio <= 0.75:
            symbols.append('▒  ')
        else:
            symbols.append('█  ')
    print(f"  {name:<8}" + "".join(symbols))

Heatmap matrix shape: (6, 24)
[[ 54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7
   16   8  13  11   0  56]
 [  0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27
   25  32  23  14   9   8]
 [  0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47
   62  50  45  27  28  30]
 [  0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40
   38  60  43  32  18   9]
 [  3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49
  105  76  41  92  60  54]
 [  0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3
    2   2   1   1   1   2]]

 ACTIVITY HEATMAP (messages by hour)
          00 03 06 09 12 15 18 21
  Aman    ▒  ▒  .  .  .  .  .  .  
  Karan   .  .  .  ░  █  ▒  ▒  ░  
  Neha    .  .  .  █  ▒  .  █  ░  
  Priya   .  .  .  █  █  ░  ▒  ░  
  Rahul   .  .  .  .  ▒  ▒  █  █  
  Vikas   .  .  .  ░  ▒  ░  ▒  ░  


## Feature 5: Top Words

In [5]:
# Word frequency across all REAL messages (system/media/deleted excluded).
# Lowercase everything, strip punctuation, skip common filler words.

STOP_WORDS = {
    'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
    'you', 'it', 'this', 'that', 'was', 'are', 'be', 'have', 'has', 'had',
    'we', 'my', 'me', 'do', 'not', 'at', 'so', 'am', 'he', 'his', 'her',
    'she', 'they', 'them', 'their', 'im', 'its', 'just', 'about', 'how',
    'what', 'when', 'where', 'why', 'who', 'which', 'with', 'from', 'as',
    'by', 'but', 'if', 'then', 'than', 'too', 'very', 'can', 'will',
    'would', 'could', 'should', 'us', 'our', 'your', 'yours', 'all',
    'some', 'no', 'yes', 'up', 'down', 'out', 'off', 'over', 'under',
    'again', 'there', 'here', 'now', 'today', 'been', 'being', 'were',
    'did', 'does', 'doing', 'an', 'one', 'get', 'got', 'like', 'still',
}
PUNCT_TO_SPACE = '.,!?"()[]{}:;-_<>*&^%$#@~`'

word_counts = {}
for m in real_messages:
    text = m['text'].lower().replace("'", "")   # don't -> dont (keeps contractions as one token)
    for ch in PUNCT_TO_SPACE:
        text = text.replace(ch, ' ')
    for word in text.split():
        if word in STOP_WORDS or word == '' or word.isdigit():
            continue
        word_counts[word] = word_counts.get(word, 0) + 1

top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

print(" THIS GROUP'S FAVOURITE WORDS")
max_count = top_words[0][1] if top_words else 1
for word, count in top_words:
    bar_len = int((count / max_count) * 20)
    print(f"   {word:<10} {'█' * bar_len} {count}")

 THIS GROUP'S FAVOURITE WORDS
   guys       ████████████████████ 318
   hai        ████████████████ 268
   everyone   ███████████ 187
   telling    ███████████ 179
   bhai       ██████████ 160
   started    █████████ 150
   scene      █████████ 145
   entire     █████████ 145
   please     ████████ 141
   anyone     ████████ 139


## Feature 6: Response Speed & Silent Streaks

In [ ]:
# (a) Average response time per person: gap between someone else's message
#     and this person's next message.
# (b) Longest silent streak per person: consecutive calendar days with zero messages.

sorted_msgs = sorted(messages, key=lambda m: m['dt'])

response_gaps = {name: [] for name in participants}
last_sender = None
last_time = None
for m in sorted_msgs:
    if last_sender is not None and m['sender'] != last_sender:
        gap_seconds = (m['dt'] - last_time).total_seconds()
        response_gaps[m['sender']].append(gap_seconds)
    last_sender = m['sender']
    last_time = m['dt']

avg_response = {}
for name, gaps in response_gaps.items():
    avg_response[name] = sum(gaps) / len(gaps) if gaps else float('inf')

fastest = min(avg_response.items(), key=lambda x: x[1])
slowest = max(avg_response.items(), key=lambda x: x[1] if x[1] != float('inf') else -1)

def format_duration(seconds):
    if seconds < 3600:
        return f"{seconds/60:.1f} minutes"
    return f"{seconds/3600:.1f} hours"

print(" RESPONSE PATTERNS")
print(f"   Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})")
print(f"   Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})")

# --- silent streaks ---
start_date = min(m['dt'] for m in messages).date()
end_date = max(m['dt'] for m in messages).date()
total_days = (end_date - start_date).days + 1
all_days = [start_date + timedelta(days=i) for i in range(total_days)]

active_days = {name: set() for name in participants}
for m in messages:
    active_days[m['sender']].add(m['dt'].date())

silent_streaks = {}
silent_ranges = {}
for name in participants:
    max_streak = 0
    cur_streak = 0
    streak_start = None
    best_range = (None, None)
    for day in all_days:
        if day not in active_days[name]:
            if cur_streak == 0:
                streak_start = day
            cur_streak += 1
            if cur_streak > max_streak:
                max_streak = cur_streak
                best_range = (streak_start, day)
        else:
            cur_streak = 0
    silent_streaks[name] = max_streak
    silent_ranges[name] = best_range

print("\n LONGEST SILENT STREAKS (consecutive days with zero messages)")
for name, streak in sorted(silent_streaks.items(), key=lambda x: x[1], reverse=True):
    if streak > 0:
        rng = silent_ranges[name]
        print(f"   {name:<8}: {streak} days ({rng[0].strftime('%d %b')} - {rng[1].strftime('%d %b')})")
    else:
        print(f"   {name:<8}: 0 days (never went silent)")

## Feature 7: Personality Archetype Detection

In [ ]:
# One scoring function per archetype, each returns a numeric score for a given person.

def per_person_real_messages():
    d = {name: [] for name in participants}
    for m in real_messages:
        d[m['sender']].append(m)
    return d

person_msgs = per_person_real_messages()          # real messages only (text-based scoring)
person_all_msgs = {name: [] for name in participants}
for m in messages:                                 # ALL messages (needed for burst/timing logic)
    person_all_msgs[m['sender']].append(m)


def score_spammer(name):
    # avg burst length: how many of this person's messages arrive back-to-back
    # (on the FULL timeline) before anyone else speaks
    bursts = []
    cur_burst = 0
    for m in sorted_msgs:
        if m['sender'] == name:
            cur_burst += 1
        else:
            if cur_burst > 0:
                bursts.append(cur_burst)
            cur_burst = 0
    if cur_burst > 0:
        bursts.append(cur_burst)
    return sum(bursts) / len(bursts) if bursts else 0.0


CARING_KEYWORDS = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you',
                    'please', 'reminder', 'drink water', "don't forget", 'reached safe']

def score_group_mom(name):
    text_blob = ' '.join(m['text'].lower() for m in person_msgs[name])
    return sum(text_blob.count(kw) for kw in CARING_KEYWORDS)


def score_night_owl(name):
    msgs = person_all_msgs[name]
    if not msgs:
        return 0.0
    night = sum(1 for m in msgs if m['dt'].hour >= 23 or m['dt'].hour < 5)
    return night / len(msgs) * 100


def score_storyteller(name):
    msgs = person_msgs[name]
    if not msgs:
        return 0.0
    total_words = sum(len(m['text'].split()) for m in msgs)
    return total_words / len(msgs)


def score_drama_queen(name):
    msgs = person_msgs[name]
    if not msgs:
        return 0.0
    count = 0
    for m in msgs:
        t = m['text']
        letters_only = ''.join(ch for ch in t if ch.isalpha())
        is_allcaps = len(letters_only) >= 3 and letters_only.isupper()
        has_double_excl = t.count('!') >= 2
        if is_allcaps or has_double_excl:
            count += 1
    return count / len(msgs) * 100


def score_ghost(name):
    active = len(active_days[name])
    return (1 - active / total_days) * 100


def score_comedian(name):
    msgs = person_msgs[name]
    if not msgs:
        return 0.0
    laugh_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
    count = sum(1 for m in msgs if any(lw in m['text'].lower() for lw in laugh_words))
    return count / len(msgs) * 100


def score_question_master(name):
    msgs = person_msgs[name]
    if not msgs:
        return 0.0
    count = sum(1 for m in msgs if m['text'].strip().endswith('?'))
    return count / len(msgs) * 100


# Each archetype pairs a scoring function with the brief's own threshold rule.
# THE COMEDIAN and THE QUESTION MASTER have no fixed cutoff in the brief - they're
# explicitly called "tiebreaker / fallback", so they're only used if nobody left
# qualifies for one of the six primary archetypes.
ARCHETYPES = [
    ('THE SPAMMER', score_spammer, lambda v: v > 3),
    ('THE GROUP MOM', score_group_mom, lambda v: v > 0),
    ('THE NIGHT OWL', score_night_owl, lambda v: v > 60),
    ('THE STORYTELLER', score_storyteller, lambda v: v > 30),
    ('THE DRAMA QUEEN', score_drama_queen, lambda v: v > 30),
    ('THE GHOST', score_ghost, lambda v: v > 60),
    ('THE COMEDIAN', score_comedian, lambda v: v > 0),
    ('THE QUESTION MASTER', score_question_master, lambda v: v > 25),
]

raw_scores = {name: {arch: fn(name) for arch, fn, _ in ARCHETYPES} for name in participants}

# Tie-breaking rule (documented, as the brief asks):
# Walk archetypes in priority order above (primary six, then the two fallbacks).
# For each archetype, give it to whichever UNASSIGNED person scores highest on it -
# but only if that score clears the threshold. This keeps assignment exclusive:
# once someone's assigned, they're out of the pool, so no two people share a label.
assignments = {}
unassigned = set(participants)

for arch, _, passes in ARCHETYPES:
    if not unassigned:
        break
    candidates = sorted(((raw_scores[name][arch], name) for name in unassigned), reverse=True)
    best_score, best_name = candidates[0]
    if passes(best_score):
        assignments[best_name] = (arch, best_score)
        unassigned.remove(best_name)

# Anyone still unassigned (shouldn't happen on this dataset) just gets their
# single highest-scoring archetype regardless of threshold.
for name in unassigned:
    best_arch = max(ARCHETYPES, key=lambda a: raw_scores[name][a[0]])[0]
    assignments[name] = (best_arch, raw_scores[name][best_arch])

print(" PERSONALITY ARCHETYPES")
for name in participants:
    arch, raw = assignments[name]
    print(f"   {name:<8} -> {arch} (raw score: {raw:.1f})")

## Feature 8: The Final Report

In [ ]:
# Combine everything above into one clean, screenshot-worthy printed report.

GROUP_NAME = "Hostel Bois 4ever"
W = 60

print("=" * W)
print(f" GROUPDNA REPORT — \"{GROUP_NAME}\"".center(W))
print(f" {total_days} days • {total_msgs} messages • {len(participants)} members".center(W))
print("=" * W)

print(f" Period       : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')}")
print(f" Busiest day  : {busiest_day[0].strftime('%d %B %Y')} ({busiest_day[1]} messages)")
print(f" Busiest hour : {busiest_hour[0]:02d}:00 - {(busiest_hour[0]+1)%24:02d}:00")

print("\n MESSAGES PER PERSON")
max_msg = sorted_people[0][1]
for name, count in sorted_people:
    pct = count / total_msgs * 100
    bar_len = int((count / max_msg) * 20)
    print(f"   {name:<8} {'█' * bar_len:<20} {count:>4} ({pct:4.1f}%)")

print("\n ACTIVITY HEATMAP (hour of day, columns 00 to 23 in steps of 3)")
print("          " + " ".join(f"{h:02d}" for h in range(0, 24, 3)))
for name, idx in person_index.items():
    row = heatmap[idx]
    row_max = row.max() if row.max() > 0 else 1
    symbols = []
    for h in range(0, 24, 3):
        ratio = row[h] / row_max
        if ratio <= 0.25:
            symbols.append('.  ')
        elif ratio <= 0.5:
            symbols.append('░  ')
        elif ratio <= 0.75:
            symbols.append('▒  ')
        else:
            symbols.append('█  ')
    tag = " <- NIGHT OWL" if assignments[name][0] == 'THE NIGHT OWL' else ""
    print(f"   {name:<8}" + "".join(symbols) + tag)

print("\n THIS GROUP'S FAVOURITE WORDS")
for word, count in top_words[:5]:
    bar_len = int((count / top_words[0][1]) * 20)
    print(f"   {word:<10} {'█' * bar_len} {count}")

print("\n RESPONSE PATTERNS")
print(f"   Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})")
print(f"   Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})")

print("\n LONGEST SILENT STREAKS")
for name, streak in sorted(silent_streaks.items(), key=lambda x: x[1], reverse=True):
    if streak > 0:
        rng = silent_ranges[name]
        print(f"   {name:<8}: {streak} days ({rng[0].strftime('%d %b')} - {rng[1].strftime('%d %b')})")
    else:
        print(f"   {name:<8}: 0 days")

print("\n PERSONALITY ARCHETYPES")
for name in participants:
    arch, raw = assignments[name]
    print(f"   {name:<8} → {arch}")

print("\n" + "=" * W)
print(" Generated by GroupDNA • Built with Python + NumPy".center(W))
print("=" * W)

## Bonus: Invented 9th Archetype — THE DEADLINE WARRIOR

In [ ]:
# BONUS ARCHETYPE (not in the brief's original 8): THE DEADLINE WARRIOR.
# Specific to Indian college/hostel life — someone whose chat is dominated by
# academic-stress talk: exams, assignments, deadlines, placements, professors.
# Detection rule: % of their real messages containing at least one of these
# keywords. This is reported as an EXTRA tag alongside their main archetype,
# not swapped in for it, since the brief's 8 primary archetypes are still
# scored and assigned separately above.

DEADLINE_KEYWORDS = ['exam', 'assignment', 'deadline', 'submission', 'project',
                      'viva', 'marks', 'professor', 'placement', 'workshop', 'signature']

def score_deadline_warrior(name):
    msgs = person_msgs[name]
    if not msgs:
        return 0.0
    count = sum(1 for m in msgs if any(kw in m['text'].lower() for kw in DEADLINE_KEYWORDS))
    return count / len(msgs) * 100

deadline_scores = {name: score_deadline_warrior(name) for name in participants}
top_deadline_person = max(deadline_scores.items(), key=lambda x: x[1])

print(" BONUS ARCHETYPE: THE DEADLINE WARRIOR")
print(f"   {top_deadline_person[0]} → THE DEADLINE WARRIOR "
      f"({top_deadline_person[1]:.1f}% of messages mention exams/deadlines/placements)")